DATA UNDERSTANDING

In [ ]:
import pandas as pd
df = pd.read_csv("E-commerce Dataset.csv")

In [ ]:
print("Dataset shape")
print(df.shape)

Dataset shape
(51290, 16)


In [ ]:
print("\nColumns Names:")
print(df.columns.tolist())


Columns Names:
['Order_Date', 'Time', 'Aging', 'Customer_Id', 'Gender', 'Device_Type', 'Customer_Login_type', 'Product_Category', 'Product', 'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping_Cost', 'Order_Priority', 'Payment_method']


In [ ]:
print("\nData Types:")
print(df.dtypes)


Data Types:
Order_Date              object
Time                    object
Aging                  float64
Customer_Id              int64
Gender                  object
Device_Type             object
Customer_Login_type     object
Product_Category        object
Product                 object
Sales                  float64
Quantity               float64
Discount               float64
Profit                 float64
Shipping_Cost          float64
Order_Priority          object
Payment_method          object
dtype: object


In [ ]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
Order_Date             0
Time                   0
Aging                  1
Customer_Id            0
Gender                 0
Device_Type            0
Customer_Login_type    0
Product_Category       0
Product                0
Sales                  1
Quantity               2
Discount               1
Profit                 0
Shipping_Cost          1
Order_Priority         2
Payment_method         0
dtype: int64


In [ ]:
print("\nDuplicate Rows:")
print(df.duplicated().sum())


Duplicate Rows:
0


PREPROCESSING

In [ ]:
#calculating sales median to fill missing value
sales_median = df["Sales"].median()
print("Sales Median:", sales_median)
#filling missing values with median
df["Sales"] = df["Sales"].fillna(sales_median)

Sales Median: 133.0


In [ ]:
#quantity missing values
quantity_median = df["Quantity"].median()
print("Quantity Median:", quantity_median)
#filling missing values
df["Quantity"] = df["Quantity"].fillna(quantity_median)

Quantity Median: 2.0


In [ ]:
#verfication of missing values
print("Missing Sales:", df["Sales"].isnull().sum())
print("Missing Quantity:", df["Quantity"].isnull().sum())

Missing Sales: 0
Missing Quantity: 0


Missing values in the Sales and Quantity fields were handled using median imputation. Other fields containing isolated missing values were retained because they were not required for the specified analysis.

In [ ]:
#calculating revenue [Revenue = Sales × Quantity]
df["Revenue"] = df["Sales"] * df["Quantity"]
df[["Sales", "Quantity", "Revenue"]].head()

,Sales,Quantity,Revenue
0,140.0,1.0,140.0
1,211.0,1.0,211.0
2,117.0,5.0,585.0
3,118.0,1.0,118.0
4,250.0,1.0,250.0


In [ ]:
#changing the order date format to date and time
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Month"] = df["Order_Date"].dt.to_period("M").astype(str)
df[["Order_Date", "Month"]].head()

,Order_Date,Month
0,2018-01-02,2018-01
1,2018-07-24,2018-07
2,2018-11-08,2018-11
3,2018-04-18,2018-04
4,2018-08-13,2018-08


EXPLORATORY DATA ANALYSIS

In [ ]:
#Monthly revenue
monthly_revenue = df.groupby("Month")["Revenue"].sum().reset_index()
monthly_revenue

,Month,Revenue
0,2018-01,961588.0
1,2018-02,841929.0
2,2018-03,1077958.0
3,2018-04,1515762.0
4,2018-05,2061079.0
5,2018-06,1613190.0
6,2018-07,2022508.0
7,2018-08,1671559.0
8,2018-09,1827908.0
9,2018-10,1886809.0


In [ ]:
#top 10 products
top_products = (
    df.groupby("Product")["Revenue"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)
top_products

,Product,Revenue
0,T - Shirts,1484528.0
1,Titak watch,1425912.0
2,Running Shoes,1358336.0
3,Formal Shoes,1310802.0
4,Jeans,1296228.0
5,Shirts,1178352.0
6,Fossil Watch,961950.0
7,Towels,862068.0
8,Sofa Covers,832032.0
9,Bed Sheets,805809.0


In [ ]:
#sales by category
category_sales = (
    df.groupby("Product_Category")["Revenue"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)
category_sales

,Product_Category,Revenue
0,Fashion,11298885.0
1,Home & Furniture,4892455.0
2,Auto & Accessories,2576827.0
3,Electronic,868178.0


In [ ]:
#category month wise sales
category_month_sales = pd.pivot_table(
    df,
    index="Product_Category",
    columns="Month",
    values="Revenue",
    aggfunc="sum",
    fill_value=0
)
category_month_sales

Month,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12
Product_Category,,,,,,,,,,,,
Auto & Accessories,118333.0,90950.0,133274.0,222238.0,265426.0,208927.0,251735.0,221880.0,239170.0,259493.0,302259.0,263142.0
Electronic,42407.0,36385.0,46688.0,68939.0,109786.0,74497.0,82215.0,72304.0,74468.0,75800.0,101925.0,82764.0
Fashion,546300.0,470984.0,599557.0,871701.0,1176301.0,958408.0,1198030.0,1002447.0,1054195.0,1054990.0,1276615.0,1089357.0
Home & Furniture,254548.0,243610.0,298439.0,352884.0,509566.0,371358.0,490528.0,374928.0,460075.0,496526.0,551882.0,488111.0


exporting data

In [ ]:
df.to_csv("ecommerce_cleaned.csv", index=False)